# Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import urllib.request
from google.cloud import bigquery
from google.oauth2 import service_account


# Autenticação do Colab com o Bigquery

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
#@title Preenchimento do código do projeto
#@markdown Preencha abaixo o código do teu projeto na GCP. <br/>
#@markdown Após preenchimento do código, execute essa célula.   <br/>

project_id = "fiap10dtsr-459723"  #@param {type: "string"}
#@markdown ---
clientbq = bigquery.Client(project=project_id)


A célula a seguir testa a conexão do Big Query, listando todos os datasets disponíveis.

In [4]:
for dataset in clientbq.list_datasets():
  print(dataset.dataset_id)

enem


Teste de conectividade

In [5]:
%%time
query_sql = f"""
  SELECT COUNT( HASHID ) as HASHID, COUNT( DISTINCT HASHID ) as UNIQUE_HASHID
  FROM `{project_id}.enem.enem_all`
"""
query_job = clientbq.query(query_sql)  # Make an API request.
enem_dados = query_job.to_dataframe()
display(enem_dados)

,HASHID,UNIQUE_HASHID
0,16392126,16392126


CPU times: user 96.4 ms, sys: 20.4 ms, total: 117 ms
Wall time: 4.7 s


## Teste (e template) para busca de dados

Observe na célula a seguir que:
- É declarada uma variável com a instrução SQL a ser executada no cluster de processamento (Big Query);
- É criado um processo de buscas no Big Query;
- Os dados processados são transformado em tabela e armazenado em um DataFrame `enem_dados`.



In [6]:
%%time
query_sql = f"""
  SELECT *
  FROM `{project_id}.enem.enem_all`
  ORDER BY HASHID
  LIMIT 10
"""
query_job = clientbq.query(query_sql)  # Make an API request.
enem_dados = query_job.to_dataframe()
display(enem_dados)

,HASHID,NU_INSCRICAO,NU_ANO,CO_MUNICIPIO_RESIDENCIA,NO_MUNICIPIO_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_RESIDENCIA,NU_IDADE,TP_SEXO,TP_ESTADO_CIVIL,...,Q020,Q021,Q022,Q023,Q024,Q025,Q026,Q027,IN_TEMPO_ADICIONAL,TP_FAIXA_ETARIA
0,-9223369298222348894,190001663889,2019,3167301,Silveirânia,31,MG,19,F,1,...,A,A,C,A,B,B,None,None,0,<NA>
1,-9223368731096442187,190003872700,2019,4315602,Rio Grande,43,RS,28,F,2,...,A,A,C,A,B,B,None,None,0,<NA>
2,-9223366831767848627,190004702108,2019,4104303,Campo Mourão,41,PR,23,F,1,...,A,B,C,A,A,B,None,None,0,<NA>
3,-9223366213451068264,180008057216,2018,3550308,São Paulo,35,SP,20,M,0,...,B,B,C,A,B,A,B,A,<NA>,<NA>
4,-9223365585297536232,180009067312,2018,2101202,Bacabal,21,MA,16,F,0,...,A,A,C,A,A,A,B,A,<NA>,<NA>
5,-9223365365193955464,190006031474,2019,5208707,Goiânia,52,GO,18,M,1,...,A,A,D,A,B,B,None,None,0,<NA>
6,-9223361610136259214,180013201534,2018,4106902,Curitiba,41,PR,17,M,0,...,A,B,C,B,B,B,B,D,<NA>,<NA>
7,-9223361340702011403,190003379411,2019,4101507,Arapongas,41,PR,17,M,1,...,A,B,E,A,C,B,None,None,0,<NA>
8,-9223360730590141751,190002459424,2019,3304557,Rio de Janeiro,33,RJ,19,M,1,...,B,A,E,A,B,A,None,None,0,<NA>
9,-9223357860417723824,190004264313,2019,3550308,São Paulo,35,SP,18,F,1,...,B,A,D,A,A,B,None,None,0,<NA>


CPU times: user 71.4 ms, sys: 8.83 ms, total: 80.2 ms
Wall time: 4.02 s


# Carregamento dos Dados

<font color='red'> <H1> AQUI COMEÇAM AS DIFERENÇAS </H1> </font>

Não carregamos os dados completos, imagina agora carregar 15 GB de dados neste servidor de aplicação!

Vamos colocar nosso cluster de processamento para funcionar!
Materiais de referência:

- https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax
- https://cloud.google.com/bigquery/docs/reference/standard-sql/functions-and-operators


# Questões

## Exemplo 1
Qual a distribuição dos participantes por estado onde a prova foi realizada em cada ano e no total?

### Resposta total

In [7]:
query_sql = f"""
SELECT
  SG_UF_PROVA,
  COUNT( HASHID ) as HASHID
FROM `{project_id}.enem.enem_all`
GROUP BY 1
"""
query_job = clientbq.query(query_sql)  # Make an API request.
enem_dados = query_job.to_dataframe()
display(enem_dados)

,SG_UF_PROVA,HASHID
0,RJ,1110445
1,TO,159848
2,AC,118799
3,MA,673105
4,SP,2663832
5,SC,357213
6,PB,463632
7,RN,374589
8,ES,320057
9,MG,1694869


Cria coluna de percentual por estado

In [8]:
# Calcula o percentual sobre o total e armazena em uma nova coluna HASHID_perc
enem_dados["HASHID_perc"] = enem_dados["HASHID"] / enem_dados["HASHID"].sum()

# Muda o formato da nova coluna HASHID_perc para ficar percentual
enem_dados["HASHID_perc"] = enem_dados["HASHID_perc"].map("{:.2%}".format)

#Exibe os resultados
display(enem_dados)

,SG_UF_PROVA,HASHID,HASHID_perc
0,RJ,1110445,6.77%
1,TO,159848,0.98%
2,AC,118799,0.72%
3,MA,673105,4.11%
4,SP,2663832,16.25%
5,SC,357213,2.18%
6,PB,463632,2.83%
7,RN,374589,2.29%
8,ES,320057,1.95%
9,MG,1694869,10.34%


In [9]:
px.bar(enem_dados.sort_values("HASHID"),
       y = "HASHID",
       x = "SG_UF_PROVA",
       hover_data=["HASHID", "HASHID_perc"]
       )

In [10]:
px.pie(enem_dados,
       values="HASHID",
       names="SG_UF_PROVA")

### Resposta por ano

In [11]:
query_sql = f"""
SELECT
  SG_UF_PROVA, NU_ANO,
  COUNT( HASHID ) as HASHID
FROM `{project_id}.enem.enem_all`
GROUP BY 1, 2
"""
query_job = clientbq.query(query_sql)  # Make an API request.
enem_dados = query_job.to_dataframe()
display(enem_dados)

,SG_UF_PROVA,NU_ANO,HASHID
0,DF,2020,116932
1,PB,2018,151491
2,MA,2020,238272
3,PR,2018,237343
4,MT,2019,88121
...,...,...,...
76,GO,2020,211069
77,AL,2018,87978
78,PE,2019,275318
79,AP,2018,40622


In [12]:
# Agrupamento por ano
enem_sample_group_ano = enem_dados.groupby(["NU_ANO"]).agg({'HASHID': 'sum'}).reset_index() #Observe a mudança de COUNT para SUM
display(enem_sample_group_ano)

,NU_ANO,HASHID
0,2018,5513747
1,2019,5095270
2,2020,5783109


In [13]:
# JOIN / MERGE das duas tabelas, e já converte o ano para string
enem_sample_final = pd.merge(enem_dados, enem_sample_group_ano, on="NU_ANO", how='left', suffixes = ['', '_ano'])
enem_sample_final['NU_ANO'] = enem_sample_final['NU_ANO'].astype(str)
display(enem_sample_final)

,SG_UF_PROVA,NU_ANO,HASHID,HASHID_ano
0,DF,2020,116932,5783109
1,PB,2018,151491,5513747
2,MA,2020,238272,5783109
3,PR,2018,237343,5513747
4,MT,2019,88121,5095270
...,...,...,...,...
76,GO,2020,211069,5783109
77,AL,2018,87978,5513747
78,PE,2019,275318,5095270
79,AP,2018,40622,5513747


In [14]:
# Calcula o percentual sobre o total e armazena em uma nova coluna HASHID_perc
enem_sample_final["HASHID_perc"]     = enem_sample_final["HASHID"] / enem_dados["HASHID"].sum()
enem_sample_final["HASHID_perc_ano"] = enem_sample_final["HASHID"] / enem_sample_final["HASHID_ano"]

# Muda o formato da nova coluna HASHID_perc para ficar percentual
enem_sample_final["HASHID_perc_ano"] = enem_sample_final["HASHID_perc_ano"].map("{:.2%}".format)
enem_sample_final["HASHID_perc"] = enem_sample_final["HASHID_perc"].map("{:.2%}".format)

#Exibe os resultados
display(enem_sample_final)

,SG_UF_PROVA,NU_ANO,HASHID,HASHID_ano,HASHID_perc,HASHID_perc_ano
0,DF,2020,116932,5783109,0.71%,2.02%
1,PB,2018,151491,5513747,0.92%,2.75%
2,MA,2020,238272,5783109,1.45%,4.12%
3,PR,2018,237343,5513747,1.45%,4.30%
4,MT,2019,88121,5095270,0.54%,1.73%
...,...,...,...,...,...,...
76,GO,2020,211069,5783109,1.29%,3.65%
77,AL,2018,87978,5513747,0.54%,1.60%
78,PE,2019,275318,5095270,1.68%,5.40%
79,AP,2018,40622,5513747,0.25%,0.74%


In [15]:
px.bar(enem_sample_final.sort_values(["NU_ANO", "HASHID"]),
       y = "HASHID",
       x = "SG_UF_PROVA",
       color="NU_ANO",
       barmode='relative',
       hover_data=["HASHID", "HASHID_perc", "HASHID_perc_ano"]
       )

In [16]:
px.bar(enem_sample_final.sort_values(["NU_ANO", "HASHID"]),
       y = "HASHID",
       x = "SG_UF_PROVA",
       color="NU_ANO",
       barmode='group',
       hover_data=["HASHID", "HASHID_perc", "HASHID_perc_ano"]
       )

## Exemplo 2
Considerando apenas 2018, quais as métricas globais de média, mediana, primeiro quartil (25%), terceiro quartil (75%) dos participantes em matemática `NU_NOTA_MT`?


In [17]:
query_sql = f"""
WITH percentiles as
(     SELECT DISTINCT  e.NU_ANO,
         COUNT(HASHID) OVER (PARTITION BY e.NU_ANO) AS HASHID,
        PERCENTILE_DISC(NU_NOTA_MT, 0.03) OVER(PARTITION BY e.NU_ANO) AS p03,
        PERCENTILE_DISC(NU_NOTA_MT, 0.25) OVER(PARTITION BY e.NU_ANO) AS q1,
        PERCENTILE_DISC(NU_NOTA_MT, 0.50) OVER(PARTITION BY e.NU_ANO) AS median,
        PERCENTILE_DISC(NU_NOTA_MT, 0.75) OVER(PARTITION BY e.NU_ANO) AS q3,
        PERCENTILE_DISC(NU_NOTA_MT, 0.97) OVER(PARTITION BY e.NU_ANO) AS p97
      FROM `{project_id}.enem.enem_all` AS e
      WHERE e.NU_ANO = 2018
      AND e.TP_PRESENCA_MT = 1 )

SELECT e.NU_ANO,
      min(e.NU_NOTA_MT) as min,
      avg(e.NU_NOTA_MT) as mean,
      max(e.NU_NOTA_MT) as max,
      STDDEV(e.NU_NOTA_MT) as std,
      p.HASHID as count,
      p.p03,
      p.q1,
      p.median,
      p.q3,
      p.p97
FROM  `{project_id}.enem.enem_all` AS e
JOIN percentiles as p
ON p.NU_ANO = e.NU_ANO
GROUP BY e.NU_ANO,
      p.HASHID,
      p.p03,
      p.q1,
      p.median,
      p.q3,
      p.p97
"""
query_job = clientbq.query(query_sql)  # Make an API request.
enem_dados_agg = query_job.to_dataframe()
display(enem_dados_agg)

,NU_ANO,min,mean,max,std,count,p03,q1,median,q3,p97
0,2018,0.0,535.405566,996.1,103.151243,3905099,388.6,455.3,516.6,600.7,752.5


In [18]:
fig = go.Figure()
fig.add_trace(go.Box(orientation="h"))
fig.update_traces(q1=[enem_dados_agg.loc[0,"q1"]],
                  median=[enem_dados_agg.loc[0,"median"]],
                  q3=[enem_dados_agg.loc[0,"q3"]],
                  lowerfence=[enem_dados_agg.loc[0,"min"]],
                  upperfence=[enem_dados_agg.loc[0,"max"]], )

fig.show()

## Exemplo 3
Faça a mesma análise, porém considerando quebras por estado (local da prova).
Indique quais estados tem métricas superiores e inferiores do que as métricas nacionais.


In [19]:
query_sql = f"""
WITH percentiles as
(      SELECT DISTINCT  e.NU_ANO, e.SG_UF_PROVA,
         COUNT(HASHID) OVER (PARTITION BY e.NU_ANO, e.SG_UF_PROVA) AS HASHID,
        PERCENTILE_DISC(NU_NOTA_MT, 0.03) OVER(PARTITION BY e.NU_ANO, e.SG_UF_PROVA) AS p03,
        PERCENTILE_DISC(NU_NOTA_MT, 0.25) OVER(PARTITION BY e.NU_ANO, e.SG_UF_PROVA) AS q1,
        PERCENTILE_DISC(NU_NOTA_MT, 0.50) OVER(PARTITION BY e.NU_ANO, e.SG_UF_PROVA) AS median,
        PERCENTILE_DISC(NU_NOTA_MT, 0.75) OVER(PARTITION BY e.NU_ANO, e.SG_UF_PROVA) AS q3,
        PERCENTILE_DISC(NU_NOTA_MT, 0.97) OVER(PARTITION BY e.NU_ANO, e.SG_UF_PROVA) AS p97
      FROM `{project_id}.enem.enem_all` AS e
      WHERE e.NU_ANO = 2018
      AND e.TP_PRESENCA_MT = 1
)

SELECT e.NU_ANO, e.SG_UF_PROVA,
      min(e.NU_NOTA_MT) as min,
      avg(e.NU_NOTA_MT) as mean,
      max(e.NU_NOTA_MT) as max,
      STDDEV(e.NU_NOTA_MT) as std,
      p.HASHID as count,
      p.p03,
      p.q1,
      p.median,
      p.q3,
      p.p97
FROM  `{project_id}.enem.enem_all` AS e
JOIN percentiles as p
ON p.NU_ANO = e.NU_ANO
AND p.SG_UF_PROVA = e.SG_UF_PROVA
GROUP BY e.NU_ANO, e.SG_UF_PROVA,
      p.HASHID,
      p.p03,
      p.q1,
      p.median,
      p.q3,
      p.p97
"""
query_job = clientbq.query(query_sql)  # Make an API request.
enem_dados_estado = query_job.to_dataframe()
enem_dados_estado.index = enem_dados_estado["SG_UF_PROVA"]
display(enem_dados_estado)

,NU_ANO,SG_UF_PROVA,min,mean,max,std,count,p03,q1,median,q3,p97
SG_UF_PROVA,,,,,,,,,,,,
MA,2018,MA,0.0,500.507309,987.9,85.961502,161558,384.2,436.9,484.6,547.4,700.8
SP,2018,SP,0.0,559.202934,996.1,107.487023,643149,395.2,474.7,543.2,636.9,771.9
GO,2018,GO,0.0,531.919035,996.1,102.004268,133825,388.0,453.6,512.9,593.9,749.7
MS,2018,MS,0.0,525.520003,977.9,99.580366,48287,387.0,448.9,506.5,585.0,742.1
PA,2018,PA,0.0,506.217060,983.1,86.534770,208864,385.3,441.6,491.3,554.8,703.0
RS,2018,RS,0.0,544.500579,989.5,101.487416,167702,391.7,465.2,529.8,614.0,749.5
AM,2018,AM,0.0,502.898226,965.8,84.401208,77093,384.8,440.2,488.9,549.5,698.0
RO,2018,RO,0.0,508.207194,891.5,86.768486,42522,385.7,442.2,493.9,559.0,702.9
PE,2018,PE,0.0,528.105573,996.1,99.113647,217030,387.8,452.0,510.2,587.5,742.8


In [20]:
fig = go.Figure()
boxplot_data = {"SG_UF_PROVA":["Brasil"],
                "p03": [enem_dados_agg.loc[0,"p03"]],
                "q1": [enem_dados_agg.loc[0,"q1"]],
                "median": [enem_dados_agg.loc[0,"median"]],
                "q3": [enem_dados_agg.loc[0,"q3"]],
                "p97": [enem_dados_agg.loc[0,"p97"]],
                "lowerfence": [enem_dados_agg.loc[0,"min"]],
                "upperfence": [enem_dados_agg.loc[0,"max"]],
                }

for estado in enem_dados_estado.sort_values(["median"])["SG_UF_PROVA"]:
  mydata = enem_dados_estado.loc[estado, :]

  boxplot_data["SG_UF_PROVA"].append(estado)
  boxplot_data["q1"].append(mydata["q1"])
  boxplot_data["median"].append(mydata["median"])
  boxplot_data["q3"].append(mydata["q3"])
  boxplot_data["lowerfence"].append(mydata["min"]) # Experimente com p03
  boxplot_data["upperfence"].append(mydata["max"]) # Experimente com p97

fig.add_trace(go.Box())
fig.update_traces(x=boxplot_data["SG_UF_PROVA"],
                  q1=boxplot_data["q1"],
                  median=boxplot_data["median"],
                  q3=boxplot_data["q3"],
                  lowerfence=boxplot_data["lowerfence"],
                  upperfence=boxplot_data["upperfence"])
fig.update_layout(height=650)
fig.show()

# ATIVIDADES

<font color="red">Acompanhar slide com enunciados</font>

## Exercício 1
Faça análise análoga à realizada por estado, porém observe como a escolaridade da mãe influencia nas notas do ENEM (Q002). Comente sua análise!

In [30]:
query_sql = f"""
WITH percentiles AS (
    SELECT DISTINCT  e.Q002,
           COUNT(HASHID) OVER (PARTITION BY  e.Q002) AS HASHID,
           PERCENTILE_DISC(NU_NOTA_MT, 0.03) OVER(PARTITION BY  e.Q002) AS p03,
           PERCENTILE_DISC(NU_NOTA_MT, 0.25) OVER(PARTITION BY  e.Q002) AS q1,
           PERCENTILE_DISC(NU_NOTA_MT, 0.50) OVER(PARTITION BY  e.Q002) AS median,
           PERCENTILE_DISC(NU_NOTA_MT, 0.75) OVER(PARTITION BY  e.Q002) AS q3,
           PERCENTILE_DISC(NU_NOTA_MT, 0.97) OVER(PARTITION BY  e.Q002) AS p97
    FROM `{project_id}.enem.enem_all` AS e
    WHERE e.TP_PRESENCA_MT = 1
      AND e.Q002 IS NOT NULL
)
SELECT  e.Q002,
       MIN(e.NU_NOTA_MT) AS min_nota,
       AVG(e.NU_NOTA_MT) AS mean_nota,
       MAX(e.NU_NOTA_MT) AS max_nota,
       STDDEV(e.NU_NOTA_MT) AS std_nota,
       p.HASHID AS count,
       p.p03, p.q1, p.median, p.q3, p.p97
FROM `{project_id}.enem.enem_all` AS e
JOIN percentiles AS p
ON p.Q002 = e.Q002
GROUP BY  e.Q002,
         p.HASHID, p.p03, p.q1, p.median, p.q3, p.p97
ORDER BY e.Q002;
"""
query_job = clientbq.query(query_sql)
enem_dados_escolaridade = query_job.to_dataframe()
display(enem_dados_escolaridade)

# Gráfico de dispersão
px.scatter(enem_dados_escolaridade, x="Q002", y="mean_nota",
           title="Influência da Escolaridade da Mãe na Nota Máxima do ENEM",
           labels={"Q002": "Escolaridade da Mãe", "max_nota": "Nota Máxima em Matemática"},
           hover_data=["min_nota", "mean_nota", "max_nota"])


,Q002,min_nota,mean_nota,max_nota,std_nota,count,p03,q1,median,q3,p97
0,A,0.0,464.684212,982.5,77.595639,334967,361.0,406.8,449.5,507.5,641.7
1,B,0.0,484.573164,996.1,85.635635,1521326,365.4,419.5,469.3,536.5,672.8
2,C,0.0,500.626700,994.4,92.669710,1297969,369.3,429.5,485.3,559.5,699.1
3,D,0.0,510.184036,996.1,96.034671,1283792,371.9,436.1,495.2,572.3,713.7
4,E,0.0,532.238796,996.1,105.436749,3399613,375.4,448.9,518.4,605.0,747.0
5,F,0.0,585.389170,996.1,120.492766,1149015,387.1,488.4,582.7,676.4,809.6
6,G,0.0,599.631284,996.1,122.373119,933533,390.3,502.1,601.9,691.9,821.9
7,H,0.0,491.325186,993.0,94.245801,265204,366.2,420.9,472.1,544.7,704.7


## Exercício 2
Faça análise análoga à realizada por estado, porém observe como ter computador em casa influencia nas notas do ENEM (Q024). Comente sua análise!


In [31]:
query_sql = f"""
WITH percentiles AS (
    SELECT DISTINCT  e.Q024,
           COUNT(HASHID) OVER (PARTITION BY  e.Q024) AS HASHID,
           PERCENTILE_DISC(NU_NOTA_MT, 0.03) OVER(PARTITION BY  e.Q024) AS p03,
           PERCENTILE_DISC(NU_NOTA_MT, 0.25) OVER(PARTITION BY  e.Q024) AS q1,
           PERCENTILE_DISC(NU_NOTA_MT, 0.50) OVER(PARTITION BY  e.Q024) AS median,
           PERCENTILE_DISC(NU_NOTA_MT, 0.75) OVER(PARTITION BY  e.Q024) AS q3,
           PERCENTILE_DISC(NU_NOTA_MT, 0.97) OVER(PARTITION BY  e.Q024) AS p97
    FROM `{project_id}.enem.enem_all` AS e
    WHERE e.TP_PRESENCA_MT = 1
      AND e.Q024 IS NOT NULL
)
SELECT  e.Q024,
       MIN(e.NU_NOTA_MT) AS min_nota,
       AVG(e.NU_NOTA_MT) AS mean_nota,
       MAX(e.NU_NOTA_MT) AS max_nota,
       STDDEV(e.NU_NOTA_MT) AS std_nota,
       p.HASHID AS count,
       p.p03, p.q1, p.median, p.q3, p.p97
FROM `{project_id}.enem.enem_all` AS e
JOIN percentiles AS p
ON p.Q024 = e.Q024
GROUP BY  e.Q024,
         p.HASHID, p.p03, p.q1, p.median, p.q3, p.p97
ORDER BY e.Q024;
"""
query_job = clientbq.query(query_sql)
enem_dados_computador = query_job.to_dataframe()
display(enem_dados_computador)

# Gráfico de barras para comparar número de computadores e nota média
px.bar(enem_dados_computador, x="Q024", y="mean_nota",
       title="Influência do Número de Computadores em Casa na Nota Média do ENEM (2018)",
       labels={"Q024": "Número de Computadores", "mean_nota": "Nota Média em Matemática"},
       hover_data=["min_nota", "max_nota"])


,Q024,min_nota,mean_nota,max_nota,std_nota,count,p03,q1,median,q3,p97
0,A,0.0,486.508161,996.1,86.448755,4317617,365.1,421.0,471.5,539.1,675.7
1,B,0.0,540.824522,996.1,107.229231,4592129,378.7,455.5,528.6,616.5,755.2
2,C,0.0,603.054449,996.1,118.381339,880494,395.9,510.6,607.0,691.5,816.5
3,D,0.0,636.712887,996.1,119.166423,276151,409.1,551.0,648.1,722.5,843.8
4,E,0.0,662.138440,996.1,118.519940,119028,422.4,585.1,676.1,744.6,864.8


## Exercício 3
Faça análise análoga à realizada por estado, porém observe como a renda mensal da família influencia nas notas do ENEM (Q006). Comente sua análise!


In [33]:
query_sql = f"""
WITH percentiles AS (
    SELECT DISTINCT e.Q006,
           COUNT(HASHID) OVER (PARTITION BY e.Q006) AS HASHID,
           PERCENTILE_DISC(NU_NOTA_MT, 0.03) OVER (PARTITION BY e.Q006) AS p03,
           PERCENTILE_DISC(NU_NOTA_MT, 0.25) OVER (PARTITION BY e.Q006) AS q1,
           PERCENTILE_DISC(NU_NOTA_MT, 0.50) OVER (PARTITION BY e.Q006) AS median,
           PERCENTILE_DISC(NU_NOTA_MT, 0.75) OVER (PARTITION BY e.Q006) AS q3,
           PERCENTILE_DISC(NU_NOTA_MT, 0.97) OVER (PARTITION BY e.Q006) AS p97
    FROM `{project_id}.enem.enem_all` AS e
    WHERE e.TP_PRESENCA_MT = 1
      AND e.Q006 IS NOT NULL
)
SELECT e.Q006,
       MIN(e.NU_NOTA_MT) AS min_nota,
       AVG(e.NU_NOTA_MT) AS mean_nota,
       MAX(e.NU_NOTA_MT) AS max_nota,
       STDDEV(e.NU_NOTA_MT) AS std_nota,
       p.HASHID AS count,
       p.p03, p.q1, p.median, p.q3, p.p97
FROM `{project_id}.enem.enem_all` AS e
JOIN percentiles AS p ON p.Q006 = e.Q006
GROUP BY e.Q006, p.HASHID, p.p03, p.q1, p.median, p.q3, p.p97
ORDER BY e.Q006;
"""
query_job = clientbq.query(query_sql)
enem_dados_renda = query_job.to_dataframe()
display(enem_dados_renda)

px.line(enem_dados_renda, x="Q006", y="mean_nota",
       title="Influência da Renda Mensal da Família na Nota Média do ENEM",
       labels={"Q006": "Renda Mensal Familiar", "mean_nota": "Nota Média em Matemática"},
       hover_data=["min_nota", "max_nota"])


,Q006,min_nota,mean_nota,max_nota,std_nota,count,p03,q1,median,q3,p97
0,A,0.0,469.670222,996.1,82.949120,484081,353.8,408.5,453.8,515.3,659.1
1,B,0.0,480.764964,988.5,84.176734,2550419,362.8,417.4,465.9,531.0,666.7
2,C,0.0,503.649656,996.1,90.494406,2312901,373.7,434.0,489.8,561.6,696.3
3,D,0.0,524.415166,996.1,99.439062,1037970,374.0,446.4,513.1,593.0,725.4
4,E,0.0,535.020308,996.1,100.665464,870297,381.4,455.3,524.1,605.6,736.2
5,F,0.0,553.607466,987.9,107.148692,531211,381.0,468.9,547.8,631.1,759.7
6,G,0.0,565.330834,996.1,107.869618,655062,388.7,480.0,560.3,644.7,769.5
7,H,0.0,585.362607,996.1,111.720503,427657,393.8,497.8,585.4,668.5,790.2
8,I,0.0,597.001533,996.1,113.889407,320778,397.7,508.2,599.5,682.7,802.6
9,J,0.0,609.389548,994.4,114.636958,188272,402.2,521.8,615.3,694.7,812.7


# DESAFIO para quem curte fortes emoções

Seção reservada para os corajosos de espírito

Considere que a nota final do ENEM é composta pela nota das quatro áreas mais a nota de redação, responda:

Como a escolaridade de ambos os pais SIMULTANEAMENTE influencia na nota final do ENEM?


Dica: Primeramente examine cada um dos pais individualmente, então faça uma nova análise considerando todo o casal (A+A, A+B, B+A, etc.)



CN – Ciências da Natureza
CH – Ciências Humanas
LC – Linguagens e Códigos
MT – Matemática
REDAÇÃO

In [39]:
# Consulta SQL para analisar a influência da escolaridade dos pais na nota final do ENEM
query_sql = f"""
WITH NotaFinal AS (
    SELECT
        HASHID,
        Q001,  -- Escolaridade do pai
        Q002,  -- Escolaridade da mãe
        COALESCE(NU_NOTA_CN, 0) + COALESCE(NU_NOTA_CH, 0) + COALESCE(NU_NOTA_LC, 0) + COALESCE(NU_NOTA_MT, 0) + COALESCE(NU_NOTA_REDACAO, 0) AS Nota_Final
    FROM
        `{project_id}.enem.enem_all`
    WHERE
        Q001 IS NOT NULL AND Q002 IS NOT NULL
),
EscolaridadeCombinada AS (
  SELECT
        HASHID,
        Nota_Final,
        CASE
            WHEN Q001 = 'A' THEN 'Pai_A'
            WHEN Q001 = 'B' THEN 'Pai_B'
            WHEN Q001 = 'C' THEN 'Pai_C'
            WHEN Q001 = 'D' THEN 'Pai_D'
            WHEN Q001 = 'E' THEN 'Pai_E'
            WHEN Q001 = 'F' THEN 'Pai_F'
            WHEN Q001 = 'G' THEN 'Pai_G'
            WHEN Q001 = 'H' THEN 'Pai_H'
            ELSE 'Pai_Outros'
        END as EscolaridadePai,
        CASE
            WHEN Q002 = 'A' THEN 'Mae_A'
            WHEN Q002 = 'B' THEN 'Mae_B'
            WHEN Q002 = 'C' THEN 'Mae_C'
            WHEN Q002 = 'D' THEN 'Mae_D'
            WHEN Q002 = 'E' THEN 'Mae_E'
            WHEN Q002 = 'F' THEN 'Mae_F'
            WHEN Q002 = 'G' THEN 'Mae_G'
            WHEN Q002 = 'H' THEN 'Mae_H'
            ELSE 'Mae_Outros'
            END AS EscolaridadeMae
    FROM NotaFinal
)
SELECT
    EscolaridadePai,
    EscolaridadeMae,
    AVG(Nota_Final) AS MediaNotaFinal
FROM
    EscolaridadeCombinada
GROUP BY
    EscolaridadePai, EscolaridadeMae
ORDER BY
    EscolaridadePai, EscolaridadeMae
"""

query_job = clientbq.query(query_sql)
enem_dados_escolaridade_pais = query_job.to_dataframe()
display(enem_dados_escolaridade_pais)

# Gráfico de barras para visualizar a relação
px.bar(enem_dados_escolaridade_pais, x="EscolaridadePai", y="MediaNotaFinal", color="EscolaridadeMae",
       barmode="group",
       title="Influência da Escolaridade dos Pais na Nota Final do ENEM",
       labels={"EscolaridadePai": "Escolaridade do Pai", "MediaNotaFinal": "Média da Nota Final", "EscolaridadeMae": "Escolaridade da Mãe"})


,EscolaridadePai,EscolaridadeMae,MediaNotaFinal
0,Pai_A,Mae_A,1208.581112
1,Pai_A,Mae_B,1300.673873
2,Pai_A,Mae_C,1392.454267
3,Pai_A,Mae_D,1439.994094
4,Pai_A,Mae_E,1465.254069
...,...,...,...
59,Pai_H,Mae_D,1526.917273
60,Pai_H,Mae_E,1617.283043
61,Pai_H,Mae_F,1865.623040
62,Pai_H,Mae_G,1964.422668
